# Hybrid Search for RAG

Hybrid search combines two complementary retrievers:

- **Sparse (lexical)** — BM25 over token frequencies. Strong on exact terms, names, codes, rare words.
- **Dense (semantic)** — cosine similarity over embeddings. Strong on paraphrases, synonyms, intent.

Then fuse the two rankings. Two common fusion methods:

1. **Weighted score combination** — normalize each score, then `alpha * dense + (1 - alpha) * sparse`.
2. **Reciprocal Rank Fusion (RRF)** — ignore raw scores, combine by rank: `1 / (k + rank)`.

## 1. Sparse retrieval: BM25 from scratch

For a query term `t` in document `d`:

$$\text{score}(d, t) = \text{IDF}(t) \cdot \frac{tf(t,d) \cdot (k_1 + 1)}{tf(t,d) + k_1 \cdot (1 - b + b \cdot |d| / \text{avgdl})}$$

- `k1` controls term-frequency saturation (typical: 1.2–2.0)
- `b` controls length normalization (typical: 0.75)

In [ ]:
import math
from collections import Counter

class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = []          # tokenized
        self.raw_docs = []      # original text
        self.doc_freq = {}      # term -> number of docs containing it
        self.doc_len = []
        self.avg_dl = 0
        self.N = 0

    def _tokenize(self, text):
        return text.lower().split()

    def add_documents(self, docs):
        for doc in docs:
            tokens = self._tokenize(doc)
            self.docs.append(tokens)
            self.raw_docs.append(doc)
            self.doc_len.append(len(tokens))
            for term in set(tokens):
                self.doc_freq[term] = self.doc_freq.get(term, 0) + 1
        self.N = len(self.docs)
        self.avg_dl = sum(self.doc_len) / self.N

    def _idf(self, term):
        df = self.doc_freq.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query, doc_idx):
        query_tokens = self._tokenize(query)
        tf = Counter(self.docs[doc_idx])
        dl = self.doc_len[doc_idx]
        s = 0.0
        for term in query_tokens:
            if term not in tf:
                continue
            idf = self._idf(term)
            num = tf[term] * (self.k1 + 1)
            denom = tf[term] + self.k1 * (1 - self.b + self.b * dl / self.avg_dl)
            s += idf * num / denom
        return s

    def search(self, query, top_k=3):
        scores = [(self.score(query, i), self.raw_docs[i]) for i in range(self.N)]
        scores.sort(reverse=True, key=lambda x: x[0])
        return scores[:top_k]

## 2. Dense retrieval: cosine similarity over vectors

Same shape as your `FlatIndex` in `indexing.ipynb`.

In [ ]:
import numpy as np

def cos_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

class DenseIndex:
    def __init__(self):
        self.vectors = []
        self.texts = []

    def add(self, vector, text):
        self.vectors.append(np.array(vector))
        self.texts.append(text)

    def search(self, query_vector, top_k=3):
        scores = [(cos_sim(query_vector, v), self.texts[i]) for i, v in enumerate(self.vectors)]
        scores.sort(reverse=True, key=lambda x: x[0])
        return scores[:top_k]

## 3. Hybrid search: combine BM25 + dense

Two fusion strategies are exposed:

- `weighted_search` — min-max normalize both score lists, then `alpha * dense + (1-alpha) * sparse`.
- `rrf_search` — Reciprocal Rank Fusion; robust because it ignores raw score scales.

Why normalize? BM25 scores can be unbounded; cosine is in `[-1, 1]`. Adding them raw lets BM25 dominate.

In [ ]:
class HybridSearch:
    def __init__(self, alpha=0.5):
        self.alpha = alpha          # weight on dense; (1 - alpha) on sparse
        self.bm25 = BM25()
        self.dense = DenseIndex()

    def add_documents(self, texts, vectors):
        self.bm25.add_documents(texts)
        for v, t in zip(vectors, texts):
            self.dense.add(v, t)

    @staticmethod
    def _min_max(scores):
        scores = np.array(scores, dtype=float)
        lo, hi = scores.min(), scores.max()
        if hi - lo < 1e-9:
            return np.zeros_like(scores)
        return (scores - lo) / (hi - lo)

    def weighted_search(self, query_text, query_vector, top_k=3):
        sparse = np.array([self.bm25.score(query_text, i) for i in range(self.bm25.N)])
        dense = np.array([cos_sim(query_vector, v) for v in self.dense.vectors])
        combined = self.alpha * self._min_max(dense) + (1 - self.alpha) * self._min_max(sparse)
        ranked = sorted(zip(combined, self.bm25.raw_docs), reverse=True, key=lambda x: x[0])
        return ranked[:top_k]

    def rrf_search(self, query_text, query_vector, top_k=3, k=60):
        bm25_rank = sorted(range(self.bm25.N),
                           key=lambda i: -self.bm25.score(query_text, i))
        dense_rank = sorted(range(len(self.dense.vectors)),
                            key=lambda i: -cos_sim(query_vector, self.dense.vectors[i]))
        rrf = {}
        for rank, idx in enumerate(bm25_rank):
            rrf[idx] = rrf.get(idx, 0) + 1 / (k + rank + 1)
        for rank, idx in enumerate(dense_rank):
            rrf[idx] = rrf.get(idx, 0) + 1 / (k + rank + 1)
        ranked = sorted(rrf.items(), key=lambda x: -x[1])
        return [(score, self.bm25.raw_docs[idx]) for idx, score in ranked[:top_k]]

## 4. Demo

Toy corpus with two topical clusters — ML/AI and sports — plus 3-D toy embeddings.

In [ ]:
texts = [
    "Transformer architecture for natural language processing",
    "Neural networks basics and deep learning",
    "Cricket world cup final highlights",
    "Football tactics and player positions",
    "Deep learning optimization algorithms",
]

vectors = [
    [0.95, 0.05, 0.10],
    [0.90, 0.10, 0.15],
    [0.10, 0.90, 0.05],
    [0.20, 0.80, 0.10],
    [0.85, 0.15, 0.20],
]

hs = HybridSearch(alpha=0.5)
hs.add_documents(texts, vectors)

In [ ]:
query_text = "deep learning"
query_vec  = [0.9, 0.1, 0.2]

print("BM25 only:")
for s, t in hs.bm25.search(query_text, top_k=3):
    print(f"  {s:.4f}  {t}")

print("\nDense only:")
for s, t in hs.dense.search(query_vec, top_k=3):
    print(f"  {s:.4f}  {t}")

print("\nHybrid (weighted, alpha=0.5):")
for s, t in hs.weighted_search(query_text, query_vec, top_k=3):
    print(f"  {s:.4f}  {t}")

print("\nHybrid (RRF):")
for s, t in hs.rrf_search(query_text, query_vec, top_k=3):
    print(f"  {s:.4f}  {t}")

## 5. Tuning notes

- `alpha` near **0.7–0.8** usually wins when queries are conversational; lower it (0.3–0.5) when users type keywords/IDs/code.
- RRF needs no tuning of scales — a good default when you can’t calibrate `alpha` on a dev set.
- For real corpora, replace the toy tokenizer with one that handles punctuation, stopwords, and stemming (e.g. `nltk` or `spaCy`).
- Swap `BM25` for `rank_bm25.BM25Okapi` and `DenseIndex` for FAISS / Chroma / pgvector when scale matters — the fusion logic stays identical.